# VERA: Qwen3.5-4B LoRA on Kaggle

Use a **private** notebook and Dataset, enable **Internet** and **GPU T4 x2**, and attach the prepared `vera-sft-v1` Dataset. Execute cells individually. This notebook downloads model weights and trains **only on Kaggle**. Start with one GPU; two-GPU DDP requires its own successful smoke run. The data is template-generated, not expert- or HITL-approved. The first cell verifies the bundle and its automated data-audit report; it does not ask you to claim a manual review. Source-use terms remain applicable.

In [ ]:
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

candidates = [
    p.parent for p in Path('/kaggle/input').rglob('bundle.json')
    if (p.parent / 'requirements-kaggle.txt').is_file()
]
if len(candidates) != 1:
    raise RuntimeError('Attach exactly one prepared VERA bundle Dataset.')
BUNDLE = candidates[0]
DATA = BUNDLE / 'data'
WORK = Path('/kaggle/working/vera-finetuning')
WORK.mkdir(parents=True, exist_ok=True)
manifest = json.loads((BUNDLE / 'bundle.json').read_text())
for relative, expected in manifest['files'].items():
    path = (BUNDLE / relative).resolve()
    if not path.is_relative_to(BUNDLE.resolve()):
        raise RuntimeError(f'Invalid bundle path: {relative}')
    if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
        raise RuntimeError(f'Bundle verification failed: {relative}')
quality = json.loads((BUNDLE / 'quality-report.json').read_text())
data_hash = hashlib.sha256((DATA / 'manifest.json').read_bytes()).hexdigest()
if quality['passed'] is not True or quality['dataset_sha256'] != data_hash:
    raise RuntimeError('The automated data audit is missing, failed or stale.')
print('Verified bundle:', BUNDLE)
print('Automated data audit:', quality['splits'])

## Install once, then restart the kernel

This installs the pinned CUDA 12.6/PyTorch stack and Unsloth revisions **on Kaggle**. After completion, use **Restart kernel/session**, rerun the location cell, skip the installation cell if its marker is present, and continue with configuration. Do not downgrade precision to bypass an error.

In [ ]:
marker = WORK / 'installation.json'
signature = hashlib.sha256((BUNDLE / 'bundle.json').read_bytes()).hexdigest()
if marker.exists() and json.loads(marker.read_text())['signature'] == signature:
    print('Installation already recorded; continue after a kernel restart.')
else:
    wheels = list((BUNDLE / 'wheels').glob('*.whl'))
    if len(wheels) != 1:
        raise RuntimeError('Expected exactly one VERA wheel.')
    command = [sys.executable, '-m', 'pip', 'install', '--quiet', '-r']
    command += [str(BUNDLE / 'requirements-kaggle.txt'), str(wheels[0])]
    subprocess.run(command, check=True)
    marker.write_text(json.dumps({'pid': os.getpid(), 'signature': signature}))
    print('Installation complete. Restart the kernel before continuing.')

In [ ]:
marker = WORK / 'installation.json'
if not marker.exists() or json.loads(marker.read_text())['pid'] == os.getpid():
    raise RuntimeError('Complete installation and restart the kernel first.')
signature = hashlib.sha256((BUNDLE / 'bundle.json').read_bytes()).hexdigest()
if json.loads(marker.read_text())['signature'] != signature:
    raise RuntimeError('The bundle changed; rerun installation and restart the kernel.')
GPU_COUNT = 1
MAX_LENGTH = 2048
RUN_FULL_TRAINING = False
RUN_FINAL_TEST = False
EXPORT_FORMAT = None
SMOKE_REPORT_OVERRIDE = None
RESUME_CHECKPOINT = None
if GPU_COUNT not in (1, 2):
    raise ValueError('GPU_COUNT must be 1 or 2.')
SMOKE = WORK / f'smoke-{GPU_COUNT}gpu'
FULL = WORK / f'full-{GPU_COUNT}gpu'
SMOKE_REPORT = Path(SMOKE_REPORT_OVERRIDE) if SMOKE_REPORT_OVERRIDE else SMOKE / 'report.json'
COMMON = ['--data', DATA, '--max-length', MAX_LENGTH]

def run(module, arguments, distributed=False):
    environment = dict(os.environ)
    environment['CUDA_VISIBLE_DEVICES'] = '0,1' if distributed and GPU_COUNT == 2 else '0'
    command = [sys.executable, '-m', module]
    if distributed and GPU_COUNT == 2:
        command = [sys.executable, '-m', 'torch.distributed.run', '--standalone']
        command += ['--nproc_per_node=2', '--module', module]
    subprocess.run(command + [str(item) for item in arguments], env=environment, check=True)

subprocess.run(['nvidia-smi'], check=True)

## Validate lengths and establish the baseline

Preflight downloads only the tokenizer, verifies every shard, and rejects truncation. If it fails, inspect the named example and context length rather than forcing a shorter limit. The baseline and adapter evaluations use the same stratified validation subset, without claim repair or deterministic fallback.

In [ ]:
run('vera.training.preflight', COMMON)

In [ ]:
run('vera.training.evaluate', COMMON + ['--output', WORK / 'baseline-validation.json'])

## Smoke run: finite gradients and real adapter updates

This runs 40 optimizer updates. The loader must activate its Qwen3.5 safeguards on T4; no generic FP16 override is used. Stop if there is an OOM, non-finite gradient, invalid environment, or unchanged adapter. DDP replicates the full model: it cannot solve a single-card memory failure.

In [ ]:
run('vera.training.train', COMMON + ['--output', SMOKE, '--steps', 40], distributed=True)
print((SMOKE / 'report.json').read_text())

In [ ]:
smoke_validation = WORK / f'smoke-{GPU_COUNT}gpu-validation.json'
run('vera.training.evaluate', COMMON + [
    '--adapter', SMOKE / 'adapter', '--output', smoke_validation,
])
for name in ['baseline-validation.json', smoke_validation.name]:
    report = json.loads((WORK / name).read_text())
    print(name, report['rates'], 'seconds:', report['average_seconds'])

## Explicit approval before one full epoch

Inspect the smoke report, raw answers, memory, and measured time. Then set `RUN_FULL_TRAINING = True` in the configuration cell and rerun **configuration only**, followed by this cell. Do not rerun completed training/evaluation cells: existing outputs are protected. A different GPU count, context, rank, or recipe needs a matching new smoke run. Full training starts from the pinned base, not from the smoke adapter.

In [ ]:
if RUN_FULL_TRAINING:
    arguments = COMMON + [
        '--output', FULL, '--mode', 'train', '--smoke-report', SMOKE_REPORT,
    ]
    if RESUME_CHECKPOINT:
        arguments += ['--resume', RESUME_CHECKPOINT]
    run('vera.training.train', arguments, distributed=True)
else:
    print('Full training disabled. Review the smoke results before enabling it.')

In [ ]:
if RUN_FULL_TRAINING:
    run('vera.training.evaluate', COMMON + [
        '--adapter', FULL / 'adapter',
        '--output', WORK / f'full-{GPU_COUNT}gpu-validation.json',
    ])
    print((FULL / 'report.json').read_text())

## Final held-out test, only after selecting the adapter

Set `RUN_FINAL_TEST = True` only once model selection is finished. `--limit 0` evaluates all 300 test examples. Do not tune further against these results. These checks assess JSON, grounded claim/source selection, scope and status; they do not prove all summary statements correct or replace VERA's HITL integration checks.

In [ ]:
if RUN_FINAL_TEST:
    for label, adapter in [('base', None), ('finetuned', FULL / 'adapter')]:
        arguments = COMMON + [
            '--output', WORK / f'{label}-test.json', '--split', 'test', '--limit', 0,
        ]
        if adapter:
            arguments += ['--adapter', adapter]
        run('vera.training.evaluate', arguments)
else:
    print('The held-out test remains unused.')

## Optional export and persistence

Leave `EXPORT_FORMAT = None` to save quota. Use `'merged'` or `'gguf'` only after selecting a successful adapter. Export may need substantial RAM/disk, network access, and llama.cpp compilation; MTP preservation is not verified. Evaluate the final GGUF again locally. **Download the results ZIP before ending the session**: it includes the adapter, retained checkpoints, reports and original smoke report. Save the notebook without automatically rerunning all cells. For resume, upload the extracted results as a private Dataset and set `SMOKE_REPORT_OVERRIDE` to `provenance/smoke-report.json` and `RESUME_CHECKPOINT` to a complete `checkpoint-*` directory.

In [ ]:
if EXPORT_FORMAT is not None:
    run('vera.training.export', [
        '--adapter', FULL / 'adapter', '--output', WORK / f'export-{EXPORT_FORMAT}',
        '--format', EXPORT_FORMAT,
    ])

In [ ]:
import zipfile

full_exists = (FULL / 'adapter').is_dir() or any(FULL.glob('checkpoint-*'))
selected = FULL if full_exists else SMOKE
archive = WORK.parent / f'vera-results-{selected.name}.zip'
if not (selected / 'adapter').is_dir() and not any(selected.glob('checkpoint-*')):
    raise RuntimeError('No adapter or resumable checkpoint is available to download.')
files = list((selected / 'adapter').rglob('*'))
files += [selected / name for name in ('recipe.json', 'report.json', 'environment.txt')]
files += list(WORK.glob('*validation.json')) + list(WORK.glob('*test.json'))
for checkpoint in selected.glob('checkpoint-*'):
    files += list(checkpoint.rglob('*'))
with zipfile.ZipFile(archive, 'x', compression=zipfile.ZIP_STORED) as package:
    for path in files:
        if path.is_file():
            package.write(path, str(path.relative_to(WORK)))
    if SMOKE_REPORT.is_file():
        package.write(SMOKE_REPORT, 'provenance/smoke-report.json')
    else:
        print('Smoke report missing: full training remains locked until a new smoke run succeeds.')
    package.write(DATA / 'manifest.json', 'provenance/data-manifest.json')
print('Download before ending this session:', archive)
print('The ZIP preserves checkpoints, any completed adapter and available smoke provenance.')